# Transfer Learning

bài này sẽ giới thiệu các kĩ thuật để giải qiueest khi bạn không có đủ dữ liệu cho việc training model.

## Transfer Learning

Bạn có bài toán cần nhận diện 1000 người nổi tiếng ở Việt Nam, tuy nhiên dữ liệu để train chỉ khoảng 10 ảnh/1 người. Số lượng dữ liệu là quá ít để train một mô hình CNN hoàn chỉnh.

Bạn tìm trên mạng thấy VGGFace2 dataset có 3.31 triệu ảnh của 9131 người, với trung bình 362.6 ảnh cho mỗi người. Họ làm bài toán tương tự mình đó là nhận diện ảnh từng người và họ đã train được CNN model với accuracy hơn 99%.

Bạn nhớ là trong CNN, convolutional layer có tác dụng lấy các đặc trưng của ảnh, và sau hàng loạt các convolutional layer + pooling layer (ConvNet) thì $\text{model sẽ học được các đặc điểm của ảnh}$, trước khi được cho vào fully connected layer

$\rightarrow$ ConvNet trong VGGFace2 model cũng lấy ra được các đặc điểm của mặt người (tai, mũi, tóc, ...)

$\rightarrow$ Ta cũng có thể áp dụng phần ConvNet của VGGFace2 model vào bài toán nhận diện mặt người nổi tiếng ở Việt Nam để lấy ra các đặc điểm của mặt.

Quá trình sử dụng pre-trained model như trên gọi là $\text{transfer learning}$.

![](image1.png)

Các pre-trained model được sử dụng thường là các bài toán được train với dữ liệu lớn ví dụ ImageNet, dữ liệu chứa 1.2 triệu ảnh với 1000 thể loại khác nhau.

Có $2$ loại $\text{Transfer Learning}$:
- $\text{Feature Extractor}$: sau khi lấy ra các đặc điểm của ảnh bằng việc sử dụng ConvNet của pretrained-model, thì ta sẽ dùng linear classifier (linear SVM, softmax classifier, ...) để phân loại ảnh. Hiểu đơn giản thì các dặc điểm ảnh (tai, mũi, tóc, ...) giờ như input của bài toán linear regression hay logistic regression.
- $\text{Fine Tuning}$: sau khi lấy ra các đặc điểm của ảnh bằng việc sử dụng ConvNet của pretrained-model, thì ta sẽ coi đây là input của 1 CNN mới bằng cách thêm các ConvNet và FC layer.Lý do là ConvNet của VGGFace2 model có thể lấy ra được các thuộc tính của mặt người nói chung nhưng người VN có những đặc tính khác nên cần thêm 1 số ConvNet mới để học thêm các thuộc tính của người Việt Nam.

$\text{Bài toán}$: Ta muốn nhận diện ảnh của 17 loài hoa, mỗi loài hoa có khoảng 80 ảnh.

Sử dụng pre-trained model là VGG16 của ImageNet. Mô hình VGG 16 mọi người có thể xem lại. Mô hình VGG16 của ImageNet dataset, phân loại ảnh thuộc 1000 thể loại khác nhau. Nên có thể hiểu là nó đủ tổng quát để táhc ra các đặc điểm của bức ảnh, cụ thể ở đây là hoa.

## Feature Extractor

Ta chỉ giữ lại phần ConvNet trong CNN và bỏ đi FCs. Sau đó dùng output của ConvNet còn lại để làm input cho Logistic Regression với nhiều output.

![](image2.png)

Mô hình $\text{Logistic Regression với nhiều output}$ có 2 dạng:

- $\text{Dạng thứ 1}$: là một neural network, không có hidden layer, hàm activation ở output layer là $\text{softmax function}$, loss function là hàm $\text{categorical-cross entropy}$, giống như bài phân loại ảnh.

![](image3.png)

- $\text{Dạng thứ 2}$: giống như bài logistic regression, túc là model chỉ phân loại 2 class. Mỗi lần ta sẽ phân loại 1 class với tất cả các class còn lại.

![](image4.png)

#### Example Feature Extractor:

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from imutils import paths
from keras.applications import VGG16
from keras.applications import imagenet_utils
from keras.preprocessing.image import img_to_array
from keras.preprocessing.image import load_img
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import numpy as np
import random
import os

In [ ]:
# Lấy các đường dẫn đến ảnh.
image_path = list(paths.list_images('dataset/'))

# Đổi vị trí ngẫu nhiên các đường dẫn ảnh
random.shuffle(image_path)

In [ ]:
# Đường dẫn ảnh sẽ là dataset/tên_loài_hoa/tên_ảnh ví dụ dataset/Bluebell/image_0241.jpg nên p.split(os.path.sep)[-2] sẽ lấy ra được tên loài hoa
labels = [p.split(os.path.sep)[-2] for p in image_path]

# Chuyển tên các loài hoa thành số
le = LabelEncoder()
labels = le.fit_transform(labels)

In [ ]:
# Load model VGG 16 của ImageNet dataset, include_top=False để bỏ phần Fully connected layer ở cuối.
model = VGG16(weights='imagenet', include_top=False)

In [ ]:
# Load ảnh và resize về đúng kích thước mà VGG 16 cần là (224,224)
list_image = []
for (j, imagePath) in enumerate(image_path):
    image = load_img(imagePath, target_size=(224,224))
    image = img_to_array(image)

    image = np.expand_dims(image, 0)
    image = imagenet_utils.preprocess_input(image)

    list_image.append(image)

list_image = np.vstack(list_image)

In [ ]:
# Dùng pre-trained model để lấy ra các feature của ảnh
features = model.predict(list_image)

# Giống bước flatten trong CNN, chuyển từ tensor 3 chiều sau ConvNet sang vector 1 chiều
features = features.reshape((features.shape[0], 512*7*7))

In [ ]:
# Chia training set, test set tỉ lệ 80-20
X_train, X_test, y_train, y_test = train_test_split(
    features, 
    labels, 
    test_size=0.2, 
    random_state=42
)

In [ ]:
# Grid search để tìm các parameter tốt nhất cho model. C = 1/lamda, hệ số trong regularisation. Solver là kiểu optimize
# https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
params = {'C' : [0.1, 1.0, 10.0, 100.0]}
#model = GridSearchCV(LogisticRegression(solver='lbfgs', multi_class='multinomial'), params)
model = GridSearchCV(LogisticRegression(), params)
model.fit(X_train, y_train)
print('Best parameter for the model {}'.format(model.best_params_))

Best parameter for the model {'C': 0.1}

In [ ]:
# Đánh giá model
preds = model.predict(X_test)
print(classification_report(y_test, preds))

![](image5.png)

## Fine Tuning

Ta chỉ giữ lại phần ConvNet trong CNN và bỏ đi FCs. Sau đó thêm các FC layer mới vào output của ConvNet.

![](image6.png)

Khi train model ta chia làm 2 giai đoạn

- $\text{Giai đoạn 1}$: Vì các fully connected layer ta mới thêm vào có các hệ số được khởi tạo ngẫu nhiên tuy nhiên các layer trong ConvNet của pre-trained model đã được train với ImageNet dataset nên ta sẽ không train (đóng băng/freeze) trên các layer trong ConvNet của model VGG16. Sau khoảng 20-30 epochs thì các hệ số ở các layer mới đã được học từ dữ liệu thì ta chuyển snag $\text{giai đoạn 2}$.

![](image7.png)

- $\text{Giai đoạn 2}$: ta sẽ unfreeze các layer trên ConvNet của pre-trained model và train trên các layer của ConvNet của pre-trained model và các layer mới. Bạn có thể unfreeze tất cả các layer trong ConvNet của VGG16 hoặc chỉ unfreeze một vài layer cuoois tuỳ vào thời gian và GPU bạn có.

![](image8.png)

#### Example Fine Tuning

In [ ]:
# Thêm thư viện
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from imutils import paths
from keras.applications import VGG16
from keras.applications import imagenet_utils
from keras.preprocessing.image import img_to_array
from keras.preprocessing.image import load_img
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import train_test_split
from keras.preprocessing.image import ImageDataGenerator
from keras.optimizers import SGD
from keras.optimizers import RMSprop
from keras.applications import VGG16
from keras.layers import Input
from keras.models import Model
from keras.layers.core import Dense
from keras.layers.core import Dropout
from keras.layers.core import Flatten
import numpy as np
import random
import os

In [ ]:
# Lấy các đường dẫn đến ảnh.
image_path = list(paths.list_images('dataset/'))

# Đổi vị trí ngẫu nhiên các đường dẫn ảnh
random.shuffle(image_path)

In [ ]:
# Đường dẫn ảnh sẽ là dataset/tên_loài_hoa/tên_ảnh ví dụ dataset/Bluebell/image_0241.jpg nên p.split(os.path.sep)[-2] sẽ lấy ra được tên loài hoa
labels = [p.split(os.path.sep)[-2] for p in image_path]

# Chuyển tên các loài hoa thành số
le = LabelEncoder()
labels = le.fit_transform(labels)

# One-hot encoding
lb = LabelBinarizer()
labels = lb.fit_transform(labels)

In [ ]:
# Load ảnh và resize về đúng kích thước mà VGG 16 cần là (224,224)
list_image = []
for (j, imagePath) in enumerate(image_path):
    image = load_img(imagePath, target_size=(224, 224))
    image = img_to_array(image)
    
    image = np.expand_dims(image, 0)
    image = imagenet_utils.preprocess_input(image)
    
    list_image.append(image)
    
list_image = np.vstack(list_image)

In [ ]:
# Load model VGG 16 của ImageNet dataset, include_top=False để bỏ phần Fully connected layer ở cuối.
baseModel = VGG16(
    weights='imagenet',
    include_top=False,
    input_tensor=Input(shape=(224, 224, 3))
)

# Xây thêm các layer
# Lấy output của ConvNet trong VGG16
fcHead = baseModel.output

# Flatten trước khi dùng FCs
fcHead = Flatten(name='flatten')(fcHead)

# Thêm FC
fcHead = Dense(256, activation='relu')(fcHead)
fcHead = Dropout(0.5)(fcHead)

# Output layer với softmax activation
fcHead = Dense(17, activation='softmax')(fcHead)

# Xây dựng model bằng việc nối ConvNet của VGG16 và fcHead
model = model = Model(inputs=baseModel.input, outputs=fcHead)

In [ ]:
# Chia traing set, test set tỉ lệ 80-20
X_train, X_test, y_train, y_test = train_test_split(list_image, labels, test_size=0.2, random_state=42)

In [ ]:
# augmentation cho training data
aug_train = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# augementation cho test
# KHÔNG augmentation
# => Test phải phản ánh dữ liệu thật
aug_test= ImageDataGenerator(rescale=1./255)

In [ ]:
# freeze VGG model
for layer in baseModel.layers:
    layer.trainable = False

otp = RMSprop(0.001)
model.compile(otp, 'categorical_crossentropy', ['accuracy'])
numOfEpoch = 25
H = model.fit_generator(
    aug_train.flow(X_train, y_train, batch_size=32),
    steps_per_epoch=len(X_train)//32,
    validation_data=(aug_test.flow(X_test, y_test, batch_size=32)),
    validation_steps=len(X_test)//32,
    epochs=numOfEpoch
)


Epoch 1/25
34/34 [==============================] - 36s 1s/step - loss: 5.8708 - acc: 0.1443 - val_loss: 1.8312 - val_acc: 0.4805
Epoch 2/25
34/34 [==============================] - 25s 727ms/step - loss: 2.2167 - acc: 0.3097 - val_loss: 1.4920 - val_acc: 0.4833
Epoch 3/25
34/34 [==============================] - 21s 631ms/step - loss: 1.8023 - acc: 0.4292 - val_loss: 1.2073 - val_acc: 0.6583
Epoch 4/25
34/34 [==============================] - 21s 632ms/step - loss: 1.5419 - acc: 0.5064 - val_loss: 0.9879 - val_acc: 0.7208
Epoch 5/25
34/34 [==============================] - 22s 634ms/step - loss: 1.4261 - acc: 0.5506 - val_loss: 0.6725 - val_acc: 0.8083
Epoch 6/25
34/34 [==============================] - 22s 633ms/step - loss: 1.3022 - acc: 0.5873 - val_loss: 0.7243 - val_acc: 0.7667
Epoch 7/25
34/34 [==============================] - 22s 636ms/step - loss: 1.1401 - acc: 0.6241 - val_loss: 0.6351 - val_acc: 0.8708
Epoch 8/25
34/34 [==============================] - 22s 636ms/step - loss: 1.1156 - acc: 0.6480 - val_loss: 0.5055 - val_acc: 0.8500
Epoch 9/25
34/34 [==============================] - 22s 636ms/step - loss: 1.0335 - acc: 0.6498 - val_loss: 0.4785 - val_acc: 0.8417
Epoch 10/25
34/34 [==============================] - 22s 638ms/step - loss: 0.9727 - acc: 0.6949 - val_loss: 0.4631 - val_acc: 0.8711
Epoch 11/25
34/34 [==============================] - 22s 638ms/step - loss: 0.9333 - acc: 0.7050 - val_loss: 0.5691 - val_acc: 0.8417
Epoch 12/25
34/34 [==============================] - 22s 638ms/step - loss: 0.8679 - acc: 0.7243 - val_loss: 0.5470 - val_acc: 0.8125
Epoch 13/25
34/34 [==============================] - 22s 639ms/step - loss: 0.8397 - acc: 0.7335 - val_loss: 0.5751 - val_acc: 0.8000
Epoch 14/25
34/34 [==============================] - 22s 639ms/step - loss: 0.8081 - acc: 0.7491 - val_loss: 0.5523 - val_acc: 0.8333
Epoch 15/25
34/34 [==============================] - 22s 643ms/step - loss: 0.7724 - acc: 0.7629 - val_loss: 0.4636 - val_acc: 0.8750
Epoch 16/25
34/34 [==============================] - 22s 640ms/step - loss: 0.7494 - acc: 0.7509 - val_loss: 0.3266 - val_acc: 0.9083
Epoch 17/25
34/34 [==============================] - 22s 641ms/step - loss: 0.7434 - acc: 0.7445 - val_loss: 0.4980 - val_acc: 0.8625
Epoch 18/25
34/34 [==============================] - 22s 638ms/step - loss: 0.6952 - acc: 0.7693 - val_loss: 0.3791 - val_acc: 0.8917
Epoch 19/25
34/34 [==============================] - 22s 637ms/step - loss: 0.6805 - acc: 0.7739 - val_loss: 0.4193 - val_acc: 0.8633
Epoch 20/25
34/34 [==============================] - 22s 640ms/step - loss: 0.6195 - acc: 0.7858 - val_loss: 0.4143 - val_acc: 0.8750
Epoch 21/25
34/34 [==============================] - 22s 639ms/step - loss: 0.6625 - acc: 0.7849 - val_loss: 0.3923 - val_acc: 0.8958
Epoch 22/25
34/34 [==============================] - 22s 636ms/step - loss: 0.5939 - acc: 0.8079 - val_loss: 0.4184 - val_acc: 0.8833
Epoch 23/25
34/34 [==============================] - 22s 643ms/step - loss: 0.5655 - acc: 0.8171 - val_loss: 0.3251 - val_acc: 0.9042
Epoch 24/25
34/34 [==============================] - 22s 637ms/step - loss: 0.6053 - acc: 0.8079 - val_loss: 0.5748 - val_acc: 0.8542
Epoch 25/25
34/34 [==============================] - 22s 636ms/step - loss: 0.5360 - acc: 0.8208 - val_loss: 0.3918 - val_acc: 0.9000

In [ ]:
# unfreeze some last CNN layer:
for layer in baseModel.layers[15:]:
    layer.trainable = True

numOfEpoch = 35
opt = SGD(0.001)
model.compile(opt, 'categorical_crossentropy', ['accuracy'])
H = model.fit_generator(
    aug_train.flow(X_train, y_train, batch_size=32), 
    steps_per_epoch=len(X_train)//32,
    validation_data=(aug_test.flow(X_test, y_test, batch_size=32)),
    validation_steps=len(X_test)//32,
    epochs=numOfEpoch
)

Epoch 1/35
34/34 [==============================] - 26s 771ms/step - loss: 0.4242 - acc: 0.8603 - val_loss: 0.3123 - val_acc: 0.9258
Epoch 2/35
34/34 [==============================] - 24s 712ms/step - loss: 0.3769 - acc: 0.8814 - val_loss: 0.2623 - val_acc: 0.9250
Epoch 3/35
34/34 [==============================] - 24s 709ms/step - loss: 0.3101 - acc: 0.9062 - val_loss: 0.2925 - val_acc: 0.9375
Epoch 4/35
34/34 [==============================] - 24s 704ms/step - loss: 0.2902 - acc: 0.9035 - val_loss: 0.4253 - val_acc: 0.9292
Epoch 5/35
34/34 [==============================] - 24s 705ms/step - loss: 0.2829 - acc: 0.8961 - val_loss: 0.3271 - val_acc: 0.9417
Epoch 6/35
34/34 [==============================] - 24s 707ms/step - loss: 0.2556 - acc: 0.9191 - val_loss: 0.1768 - val_acc: 0.9250
Epoch 7/35
34/34 [==============================] - 24s 706ms/step - loss: 0.2351 - acc: 0.9228 - val_loss: 0.3223 - val_acc: 0.9375
Epoch 8/35
34/34 [==============================] - 24s 707ms/step - loss: 0.2641 - acc: 0.9118 - val_loss: 0.3800 - val_acc: 0.9292
Epoch 9/35
34/34 [==============================] - 24s 708ms/step - loss: 0.2760 - acc: 0.9173 - val_loss: 0.2979 - val_acc: 0.9458
Epoch 10/35
34/34 [==============================] - 24s 710ms/step - loss: 0.2466 - acc: 0.9090 - val_loss: 0.3352 - val_acc: 0.9453
Epoch 11/35
34/34 [==============================] - 24s 709ms/step - loss: 0.2247 - acc: 0.9210 - val_loss: 0.2563 - val_acc: 0.9500
Epoch 12/35
34/34 [==============================] - 24s 709ms/step - loss: 0.2423 - acc: 0.9274 - val_loss: 0.3930 - val_acc: 0.9417
Epoch 13/35
34/34 [==============================] - 24s 709ms/step - loss: 0.2042 - acc: 0.9311 - val_loss: 0.2740 - val_acc: 0.9542
Epoch 14/35
34/34 [==============================] - 24s 709ms/step - loss: 0.2141 - acc: 0.9256 - val_loss: 0.3703 - val_acc: 0.9375
Epoch 15/35
34/34 [==============================] - 24s 705ms/step - loss: 0.2355 - acc: 0.9191 - val_loss: 0.2008 - val_acc: 0.9625
Epoch 16/35
34/34 [==============================] - 24s 707ms/step - loss: 0.2310 - acc: 0.9182 - val_loss: 0.2346 - val_acc: 0.9500
Epoch 17/35
34/34 [==============================] - 24s 708ms/step - loss: 0.1893 - acc: 0.9329 - val_loss: 0.3341 - val_acc: 0.9542
Epoch 18/35
34/34 [==============================] - 24s 709ms/step - loss: 0.2123 - acc: 0.9256 - val_loss: 0.3322 - val_acc: 0.9458
Epoch 19/35
34/34 [==============================] - 24s 710ms/step - loss: 0.1802 - acc: 0.9375 - val_loss: 0.2476 - val_acc: 0.9570
Epoch 20/35
34/34 [==============================] - 24s 705ms/step - loss: 0.1902 - acc: 0.9366 - val_loss: 0.3922 - val_acc: 0.9375
Epoch 21/35
34/34 [==============================] - 24s 708ms/step - loss: 0.1852 - acc: 0.9384 - val_loss: 0.2375 - val_acc: 0.9583
Epoch 22/35
34/34 [==============================] - 24s 709ms/step - loss: 0.1782 - acc: 0.9458 - val_loss: 0.2803 - val_acc: 0.9500
Epoch 23/35
34/34 [==============================] - 24s 709ms/step - loss: 0.2021 - acc: 0.9347 - val_loss: 0.3591 - val_acc: 0.9208
Epoch 24/35
34/34 [==============================] - 24s 705ms/step - loss: 0.2031 - acc: 0.9265 - val_loss: 0.4288 - val_acc: 0.9292
Epoch 25/35
34/34 [==============================] - 24s 707ms/step - loss: 0.1587 - acc: 0.9449 - val_loss: 0.2866 - val_acc: 0.9333
Epoch 26/35
34/34 [==============================] - 24s 708ms/step - loss: 0.1837 - acc: 0.9375 - val_loss: 0.3268 - val_acc: 0.9417
Epoch 27/35
34/34 [==============================] - 24s 708ms/step - loss: 0.1921 - acc: 0.9384 - val_loss: 0.2889 - val_acc: 0.9458
Epoch 28/35
34/34 [==============================] - 24s 708ms/step - loss: 0.1747 - acc: 0.9375 - val_loss: 0.3303 - val_acc: 0.9492
Epoch 29/35
34/34 [==============================] - 24s 705ms/step - loss: 0.1627 - acc: 0.9485 - val_loss: 0.3628 - val_acc: 0.9500
Epoch 30/35
34/34 [==============================] - 24s 706ms/step - loss: 0.1721 - acc: 0.9393 - val_loss: 0.2802 - val_acc: 0.9333
Epoch 31/35
34/34 [==============================] - 24s 707ms/step - loss: 0.1788 - acc: 0.9421 - val_loss: 0.2832 - val_acc: 0.9375
Epoch 32/35
34/34 [==============================] - 24s 707ms/step - loss: 0.1880 - acc: 0.9375 - val_loss: 0.3612 - val_acc: 0.9417
Epoch 33/35
34/34 [==============================] - 24s 708ms/step - loss: 0.1932 - acc: 0.9320 - val_loss: 0.4107 - val_acc: 0.9417
Epoch 34/35
34/34 [==============================] - 24s 708ms/step - loss: 0.1603 - acc: 0.9485 - val_loss: 0.2739 - val_acc: 0.9500
Epoch 35/35
34/34 [==============================] - 24s 707ms/step - loss: 0.1168 - acc: 0.9632 - val_loss: 0.1849 - val_acc: 0.9667

Accuracy của fine-tuning tốt hơn so với feature extractor tuy nhiên thời gian train của. fine-tuning cũng lâu hơn rất nhiều. Giải thích đơn giản thì feature extractor chỉ lấy ra đặc điểm chung từ pre-trained model của ImageNet dataset cho các loài hoa, nên không được chính xác lắm. Tuy nhiên ở phần fine-tuning ta thêm các layer mới, cũng như train lại 1 số layer ở trong ConvNet của VGG16 nên model giờ học được các thuộc tính, đặc điểm của các loài hoa nên độ chính xác tốt hơn.

## Khi nào nên dùng Transfer Learning

Có $2$ yếu tố quan trọng nhất để dùng transfer learning đó là:
- $\text{kích thước của dữ liệu bạn có}$
- $\text{sự tương đồng của dữ liệu giữa mô hình bạn cần train và pre-trained model}$

#### Usecases lựa chọn:

1. Dữ liệu bạn có nhỏ và tương tự với dữ liệu ở pre-trained model. Vì dữ liệu nhỏ nên nếu dùng fine-tuning thì model sẽ bị overfitting. Hơn nữa là dữ liệu tương tự nhau nên là ConvNet của pre-trained model cũng lấy ra các đặc điểm ở dữ liệu của chúng ta. Do đó nên dùng feature extractor.

2. Dữ liệu bạn có lớn và tương tự với dữ liệu ở pre-trained model. Giờ có nhiều dữ liệu ta không sợ overfitting do đó nên dùng fine-tuning.

3. Dữ liệu bạn có nhỏ nhưng khác với dữ liệu ở pre-trained model. Vì dữ liệu nhỏ nên ta lên dùng feature extractor để tránh overfitting. Tuy nhiên do dữ liệu ta có và dữ liệu ở pre-trained model khác nhau, nên không nên dùng feature extractor với toàn bộ ConvNet của pre-trained model mà chỉ dùng các layer đầu.
    - Lý do là vì các layer ở phía trước sẽ học các đặc điểm chung chung hơn (cạnh, góc,…), còn các layer phía sau trong ConvNet sẽ học các đặc điểm cụ thể hơn trong dataset (ví dụ mắt, mũi,..).

4. Dữ liệu bạn có lớn và khác với dữ liệu ở pre-trained model. Ta có thể train model từ đầu, tuy nhiên sẽ tốt hơn nếu ta khởi tạo các giá trị weight của model với giá trị của pre-trained model và sau đó train bình thường.

#### Lưu ý:

- Vì pre-trained model đã được train với kích thước ảnh cố định, nên khi dùng pre-trained model ta cần resize lại ảnh có kích ảnh bằng kích thước mà ConvNet của pre-trained model yêu cầu.
- Hệ số learning rate của ConvNet của pre-trained model nên được đặt với giá trị nhỏ vì nó đã được học ở pre-trained model nên ít cần cập nhật hơn so với các layer mới thêm.